In [1]:
# Cell 1
# First to verify GPU
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected — check Runtime > Change runtime type")

PyTorch version: 2.10.0+cu128
GPU available: True
GPU name: Tesla T4
GPU memory: 15.6 GB


In [6]:
# Cell 2
# Installing HuggingFace transformers and datasets library
!pip install transformers datasets accelerate -q

print("HuggingFace transformers and datasets library installed")

HuggingFace transformers and datasets library installed


In [7]:
# Cell 3
# ---------------------------------------------------------------------------
# Imports for RoBERTa model
# ---------------------------------------------------------------------------

import torch
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report
)
from transformers import (
    RobertaTokenizer,           # converts text to RoBERTa token IDs
    RobertaForSequenceClassification,  # RoBERTa model for classification
    TrainingArguments,          # training hyperparameters
    Trainer,                    # HuggingFace training loop
    EarlyStoppingCallback       # stops training if no improvement
)
from datasets import Dataset    # HuggingFace dataset format

# Set device to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"All imports successful")

Using device: cuda
All imports successful


In [8]:
# Cell 4
# ---------------------------------------------------------------------------
# Uploading 4 datasets from local to Colab:
#   - training_data.csv
#   - kaggle_disaster.csv
#   - crisislex_combined.csv
#   - crisisnlp_train.tsv
# ---------------------------------------------------------------------------

from google.colab import files

print("Upload the dataset files:")
print("training_data.csv,\nkaggle_disaster.csv,\ncrisislex_combined.csv,\ncrisisnlp_train.tsv")

uploaded = files.upload()

print(f"\nUploaded files: {list(uploaded.keys())}")

Upload the dataset files:
training_data.csv,
kaggle_disaster.csv,
crisislex_combined.csv,
crisisnlp_train.tsv


Saving training_data.csv to training_data.csv
Saving kaggle_disaster.csv to kaggle_disaster.csv
Saving crisislex_combined.csv to crisislex_combined.csv
Saving crisisnlp_train.tsv to crisisnlp_train.tsv

Uploaded files: ['training_data.csv', 'kaggle_disaster.csv', 'crisislex_combined.csv', 'crisisnlp_train.tsv']


In [10]:
# Extra: Confirming and Checking all unique categories labels in CrisisNLP dataset
df_check = pd.read_csv("crisisnlp_train.tsv", sep="\t")
print("All unique class labels in CrisisNLP:")
print(df_check["class_label"].value_counts())

All unique class labels in CrisisNLP:
class_label
not_humanitarian                       36071
donation_and_volunteering               5175
requests_or_needs                       4829
sympathy_and_support                    3545
infrastructure_and_utilities_damage     3539
affected_individual                     2452
caution_and_advice                      2100
injured_or_dead_people                  1942
response_efforts                         780
missing_and_found_people                 373
displaced_and_evacuations                358
Name: count, dtype: int64


In [11]:
# Cell 5
# ---------------------------------------------------------------------------
# Load and combine all datasets
# Apply the same label mapping as model_training.ipynb
# so RoBERTa trains on the same unified schema
# ---------------------------------------------------------------------------

# 1. Our own labeled test messages
df_test = pd.read_csv("training_data.csv")
df_test = df_test[["text", "urgency_level"]].copy()
df_test.columns = ["text", "urgency_label"]

# 2. Kaggle disaster tweets
df_kaggle = pd.read_csv("kaggle_disaster.csv")
df_kaggle = df_kaggle[["text", "target"]].dropna()
df_kaggle["urgency_label"] = df_kaggle["target"].map({1: "High", 0: "Low"})
df_kaggle = df_kaggle[["text", "urgency_label"]]

# 3. CrisisLex combined
df_crisislex = pd.read_csv("crisislex_combined.csv")
crisislex_map = {
    "Related and informative":          "High",
    "Related - but not informative":    "Medium",
    "Not related":                      "Low",
    "Not applicable":                   "Low",
}
df_crisislex["urgency_label"] = df_crisislex["label"].map(crisislex_map)
df_crisislex = df_crisislex[["text", "urgency_label"]].dropna()

# 4. CrisisNLP humanitarian categories
df_crisisnlp = pd.read_csv("crisisnlp_train.tsv", sep="\t")
crisisnlp_map = {
    "injured_or_dead_people":              "Critical",
    "missing_and_found_people":            "Critical",
    "displaced_and_evacuations":           "High",
    "infrastructure_and_utilities_damage": "High",
    "caution_and_advice":                  "High",
    "affected_individual":                 "High",
    "requests_or_needs":                   "Medium",
    "sympathy_and_support":                "Medium",
    "donation_and_volunteering":           "Medium",
    "response_efforts":                    "Medium",
    "not_humanitarian":                    "Low",
}
df_crisisnlp["urgency_label"] = df_crisisnlp["class_label"].map(crisisnlp_map)
df_crisisnlp = df_crisisnlp[["text", "urgency_label"]].dropna()

# Combine all datasets
df_combined = pd.concat([
    df_test,
    df_kaggle,
    df_crisislex,
    df_crisisnlp
], ignore_index=True)

# Drop nulls and empty text
df_combined = df_combined.dropna(subset=["text", "urgency_label"])
df_combined = df_combined[df_combined["text"].str.strip() != ""]

print(f"Combined dataset: {len(df_combined):,} total messages")
print(f"\nLabel distribution:")
print(df_combined["urgency_label"].value_counts())

Combined dataset: 100,485 total messages

Label distribution:
urgency_label
Low         48683
High        27416
Medium      22066
Critical     2320
Name: count, dtype: int64


Also shown in explration ipynb,
Low + High + Medium + Critical gives us
100,485 total messages

Low has the most out of all with about 48% and Critical being the lowest about 2.3 % of the entire datasets (which is the imabalnce issue)

This will contribute to class weight for RoBERTa training, else if not, the model will not predict Critical when it has to

In [13]:
# Cell 6
# ---------------------------------------------------------------------------
# Clean text and encode labels as integers which is what RoBERTa will take in (not strings)
# Label mapping:
#   Critical → 0
#   High     → 1
#   Medium   → 2
#   Low      → 3
# ---------------------------------------------------------------------------

import re

def preprocess_text(text):
    """
    Same cleaning pipeline as backend/utils/preprocess.py
    We duplicate it here since we can't import from local machine on Colab
    """
    if not text or not isinstance(text, str):
        return ""
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove @mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtag symbols but keep word
    text = re.sub(r'#(\w+)', r'\1', text)
    # Remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply preprocessing
print("Preprocessing text...")
df_combined["cleaned_text"] = df_combined["text"].apply(preprocess_text)

# Remove empty strings after cleaning
df_combined = df_combined[df_combined["cleaned_text"].str.strip() != ""]

# Encode labels as integers
label2id = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3}
id2label = {0: "Critical", 1: "High", 2: "Medium", 3: "Low"}

df_combined["label"] = df_combined["urgency_label"].map(label2id)

print(f"Preprocessing complete: {len(df_combined):,} messages")
print(f"\nLabel encoding:")
for label, idx in label2id.items():
    count = (df_combined["label"] == idx).sum()
    print(f"  {idx} = {label:<10} ({count:,} samples)")

Preprocessing text...
Preprocessing complete: 100,424 messages

Label encoding:
  0 = Critical   (2,320 samples)
  1 = High       (27,402 samples)
  2 = Medium     (22,044 samples)
  3 = Low        (48,658 samples)


100,485 messages after cleaning/ all labels properly encoded is reduced to 100,424 messages. This is due to empty strings that was removed during clean up process

In [15]:
# Cell 7:
# ---------------------------------------------------------------------------
# Split data into 80% training and 20% testing
# The same split ratio as LR and RF in model_training.ipynb
# We stratify to ensure Critical (only 2.3%) is shown in both splits
# ---------------------------------------------------------------------------

from sklearn.model_selection import train_test_split

X = df_combined["cleaned_text"].values
y = df_combined["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,        # same seed as LR/RF for fair comparison
    stratify=y              # critical for imbalanced classes
)

print(f"Training set:  {len(X_train):,} messages")
print(f"Test set:      {len(X_test):,} messages")
print(f"\nTraining label distribution:")
for label, idx in label2id.items():
    count = (y_train == idx).sum()
    print(f"  {label:<10} {count:,}")

Training set:  80,339 messages
Test set:      20,085 messages

Training label distribution:
  Critical   1,856
  High       21,922
  Medium     17,635
  Low        38,926


In [16]:
# Cell 8
# ---------------------------------------------------------------------------
# Load RoBERTa tokenizer
# The tokenizer converts text into token IDs that RoBERTa understands
# max_length=128 to match exploration ipynb findings that 0% of messages
# exceed 128 tokens
# ---------------------------------------------------------------------------

from transformers import RobertaTokenizer

print("Loading RoBERTa tokenizer...")
# used for good balance and speed
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(texts, max_length=128):
    """
    Tokenize a list of texts for RoBERTa input.
    Returns input_ids and attention_mask tensors.
    """
    return tokenizer(
        list(texts),
        padding=True,           # pad shorter sequences
        truncation=True,        # truncate longer sequences
        max_length=max_length,  # max token length
        return_tensors="pt"     # return PyTorch tensors
    )

# Test tokenizer on one sample
sample = tokenize(["Building collapsed people trapped inside need help immediately"])
print(f"Tokenizer loaded successfully")
print(f"Sample token IDs shape: {sample['input_ids'].shape}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")

Loading RoBERTa tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully
Sample token IDs shape: torch.Size([1, 10])
Vocabulary size: 50,265


Tokenizer loads with success (50,265 tokens).
HuggingFace auth not needed for base RoBERTa

In [17]:
# Cell 9
# ---------------------------------------------------------------------------
# Now we convert data to HuggingFace Dataset format
# ---------------------------------------------------------------------------

from datasets import Dataset

def create_dataset(texts, labels):
    """
    Tokenize texts and create a HuggingFace Dataset object.
    """
    # Tokenize all texts
    encodings = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors=None     # return lists not tensors for Dataset format
    )

    # Create dataset dictionary
    dataset_dict = {
        "input_ids":      encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels":         list(labels)
    }

    return Dataset.from_dict(dataset_dict)

print("Creating HuggingFace datasets...")
print("Tokenizing training set (80,339 messages)")
train_dataset = create_dataset(X_train, y_train)

print("Tokenizing test set (20,085 messages)")
test_dataset = create_dataset(X_test, y_test)

print(f"\nTrain dataset: {train_dataset}")
print(f"Test dataset:  {test_dataset}")

Creating HuggingFace datasets...
Tokenizing training set (80,339 messages)
Tokenizing test set (20,085 messages)

Train dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 80339
})
Test dataset:  Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 20085
})


With Cell 9, we trained 80,339 rows with the input_ids, attention_mask, and labels

and Tested 20,085 rows with the same features

In [18]:
# Cell 10:
# ---------------------------------------------------------------------------
# Load RoBERTa model for sequence classification
# ---------------------------------------------------------------------------

from transformers import RobertaForSequenceClassification
import torch
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

print("Loading RoBERTa model...")

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=4,         # because we have 4 urgency levels (Critical/High/Medium/Low)
    id2label=id2label,    # allows the model to output legible labels
    label2id=label2id,
)

# Moving model to GPU
model = model.to(device)

# ---------------------------------------------------------------------------
# Compute class weights for imbalanced dataset
# Critical only has 2.3% of samples, as mentioned earlier, without weights model ignores it
# ---------------------------------------------------------------------------
classes = np.array([0, 1, 2, 3])
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

print(f"Model loaded on: {next(model.parameters()).device}")
print(f"\nClass weights:")
for idx, (label, weight) in enumerate(zip(["Critical", "High", "Medium", "Low"], weights)):
    print(f"  {label:<10} → {weight:.4f}")

# Count model parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Loading RoBERTa model...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on: cuda:0

Class weights:
  Critical   → 10.8215
  High       → 0.9162
  Medium     → 1.1389
  Low        → 0.5160

Total parameters:     124,648,708
Trainable parameters: 124,648,708


-The model is running on GPU (cuda 0)
-124,648,708 parameters which is alot and RoBERTa-base model can well handle it
-Setting up class weights: Crit has 10.82 weight over the Low's 0.52 which will balance and allow the model to give more attention to the critical samples

In [19]:
# Cell 11
# ---------------------------------------------------------------------------
# Custom Trainer that applies class weights during loss calculation
# The default HuggingFace Trainer doesn't support class weights so
# we override the compute_loss method to apply them
# This is needed for our imbalanced dataset
# ---------------------------------------------------------------------------

from transformers import Trainer
import torch.nn as nn

class WeightedTrainer(Trainer):
    """
    Custom Trainer that uses weighted cross entropy loss
    to handle class imbalance in our urgency level dataset.
    """

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Extract labels from inputs
        labels = inputs.get("labels")

        # Forward pass through model
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Apply weighted cross entropy loss
        # class_weights tensor gives higher penalty for misclassifying
        # rare classes like Critical
        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss


print("WeightedTrainer defined successfully")
print(f"Class weights applied: {class_weights.tolist()}")

WeightedTrainer defined successfully
Class weights applied: [10.821524620056152, 0.9161915183067322, 1.1389141082763672, 0.5159726142883301]


Class weights:

-Crit: 10.82 - is set to highest RoBERTa will pay more attention to

-High: 0.92

-Medium: 1.14

-Low: 0.52 - lowest given the most samples

In [21]:
# Cell 12
# ---------------------------------------------------------------------------
# Setting the training hyperparameters
# Controlling how RoBERTa is fine-tuned
# Balanced for speed and accuracy on Google Colab's T4 GPU
# ---------------------------------------------------------------------------

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./roberta_triage",      # save checkpoints here
    num_train_epochs=3,                 # 3 passes through training data
                                        # more epochs = better but slower
    per_device_train_batch_size=32,     # 32 samples per GPU batch
    per_device_eval_batch_size=64,      # larger batch ok for evaluation
    warmup_steps=500,                   # gradually increase learning rate
                                        # for first 500 steps
    weight_decay=0.01,                  # L2 regularization to prevent
                                        # overfitting
    learning_rate=2e-5,                 # standard fine-tuning rate for
                                        # RoBERTa — too high causes
                                        # catastrophic forgetting
    eval_strategy="epoch",              # evaluate after each epoch
    save_strategy="epoch",              # save checkpoint after each epoch
    load_best_model_at_end=True,        # keep best checkpoint
    metric_for_best_model="f1",         # use F1 to pick best model
    logging_dir="./logs",               # tensorboard logs
    logging_steps=100,                  # log every 100 steps
    fp16=True,                          # mixed precision training
                                        # faster on T4 GPU
    report_to="none",                   # disable wandb logging
    seed=42,                            # reproducibility
)

print("Training arguments configured:")
print(f"  Epochs:         {training_args.num_train_epochs}")
print(f"  Batch size:     {training_args.per_device_train_batch_size}")
print(f"  Learning rate:  {training_args.learning_rate}")
print(f"  Weight decay:   {training_args.weight_decay}")
print(f"  FP16 training:  {training_args.fp16}")
print(f"  Warmup steps:   {training_args.warmup_steps}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments configured:
  Epochs:         3
  Batch size:     32
  Learning rate:  2e-05
  Weight decay:   0.01
  FP16 training:  True
  Warmup steps:   500


In [22]:
# Cell 13
# ---------------------------------------------------------------------------
# After each epoch to evaluate model performance
# ---------------------------------------------------------------------------

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np

# Define metrics function
def compute_metrics(eval_pred):
    """
    Compute evaluation metrics after each epoch.
    eval_pred contains model logits and true labels.
    """
    logits, labels = eval_pred

    # Convert logits to predicted class indices
    predictions = np.argmax(logits, axis=-1)

    # Returns accuracy, F1, precision, and recall
    return {
        "accuracy":  accuracy_score(labels, predictions),
        "f1":        f1_score(labels, predictions, average="weighted"),
        "precision": precision_score(labels, predictions, average="weighted"),
        "recall":    recall_score(labels, predictions, average="weighted"),
    }

print("Metrics function defined successfully")

Metrics function defined successfully


In [23]:
# Cell 14
# ---------------------------------------------------------------------------
# Now the training begins
# Initialize trainer and start fine-tuning RoBERTa
# Training should heppen on the T4 GPU (ensure runtime type is T4 GPU hardware
# accelerator is selected)
# Note: There should be a progress bar with the loss and metrics every epoch
# AND DO NOT close the Colab tab while training is running
# ---------------------------------------------------------------------------

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Starting RoBERTa fine-tuning...")
print(f"Training on {len(train_dataset):,} messages for {training_args.num_train_epochs} epochs")
print(f"This will take a while, maybe about 15-20 min or so\n")

# Start training
trainer.train()

print("\nTraining complete!")

Starting RoBERTa fine-tuning...
Training on 80,339 messages for 3 epochs
This will take a while, maybe about 15-20 min or so



Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.509271,0.566294,0.840080,0.840325,0.840831,0.840080
2,0.430694,0.477606,0.847647,0.849447,0.853148,0.847647
3,0.302498,0.484651,0.848295,0.850656,0.856652,0.848295


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Training complete!


Results as should show:
Comparing across 3 models
Logistic Regression Model (LR): 80.5% accuracy, F1 Score: 0.809

Random Forest (RF): 78% accuracy, F1 score: 0.782

RoBERTa: 84.8% accuracy, F1 Score: 0.851 (proving the best mdoel)

There was training loss that goes down (approahces 0) after each epoch for a gradient descent which confirms correct the mdoel is getting better and better with fewer errors.

RoBERTa training loss progression:
Epoch 1 with 0.509 means it was still learning

Epoch 2 with 0.43 means it is improving

Epoch 3 with 0.302 means it is well trained enough

We stop there as running the epochs more will cause overfitting and get worse with more newer data. I think epoch 3 is a good stopping point

In [25]:
# Cell 15
# We now save the fine-tuned RoBERTa model and the tokenizer
# To be placed back into the backend/models/roberta_triage/ folder

SAVE_PATH = "./roberta_triage_final"

print("Saving fine-tuned RoBERTa model...")
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Model saved to {SAVE_PATH}")
print("\nFiles saved:")
import os
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(f"{SAVE_PATH}/{f}") / 1e6
    print(f"  {f} ({size:.1f} MB)")

Saving fine-tuned RoBERTa model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./roberta_triage_final

Files saved:
  training_args.bin (0.0 MB)
  model.safetensors (498.6 MB)
  tokenizer.json (3.6 MB)
  config.json (0.0 MB)
  tokenizer_config.json (0.0 MB)


In [26]:
# Cell 16
# Downloader to ZIP the model foler and download onto your local machine
# Expect larger file so it will take some time to download

import shutil

print("Zipping model files...")
shutil.make_archive("roberta_triage_final", "zip", "./roberta_triage_final")

print("Zip created — starting download...")
from google.colab import files
files.download("roberta_triage_final.zip")
print("Download started — it should appear in Downloads folder")

Zipping model files...
Zip created — starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started — it should appear in Downloads folder


In [4]:
# Cell 17
# ---------------------------------------------------------------------------
# Test RoBERTa on our fake messages from mentioned in projec report
# Comparing against LR and RF results from model_training.ipynb
# ---------------------------------------------------------------------------

import torch

test_messages = [
    ("There is a strong smell of gas coming from the basement of the apartment complex on 5th Ave. People are starting to feel dizzy.", "Critical"),
    ("I can see smoke and orange flames on the ridge behind the high school. The wind is picking up and blowing towards the houses.", "High"),
    ("I have run out of my prescription heart medication and the roads are blocked by snow.", "Medium"),
    ("I've had a scratchy throat and a mild cough for two days. No fever or trouble breathing. Are there any local clinics open today?", "Low"),
]

print("=" * 70)
print("Testing RoBERTa on fake messages from doc section 3.6")
print("=" * 70)

model.eval()

for msg, expected in test_messages:
    # Preprocess
    cleaned = preprocess_text(msg)

    # Tokenize
    inputs = tokenizer(
        cleaned,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    ).to(device)

    # Inference
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        predicted_id = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][predicted_id].item()

    predicted_label = id2label[predicted_id]
    match = "✓" if predicted_label == expected else "✗"

    print(f"\nMessage:    {msg[:70]}...")
    print(f"Expected:   {expected}")
    print(f"RoBERTa:    {predicted_label} (confidence: {confidence:.2f}) {match}")

Testing RoBERTa on fake messages from doc section 3.6


NameError: name 'model' is not defined